# Satellite Imagery: A Case Study in Applied Transfer Learning

**Prerequisites**

- L11 notebook 02: Vision foundation models (loading, linear probes, full fine-tuning)

**Outcomes**

- Understand how satellite imagery is used as a data source in empirical economics
- Load and explore the EuroSAT land-use classification dataset
- Fine-tune a pretrained ResNet to classify Sentinel-2 satellite imagery
- Implement **Grad-CAM** from scratch to visualize which regions of an image drive the model's prediction
- Sketch the pipeline for going from a trained classifier to an economic-activity estimate

> **Note.** The first time this notebook runs it downloads the EuroSAT dataset (~94 MB) to `./data/eurosat`. After the first run this notebook is fast to execute.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Satellite Imagery in Empirical Economics

Over the past decade, satellite imagery has become an increasingly important data source for empirical economics. The appeal is simple: satellites collect consistent, high-frequency, global coverage of the Earth's surface, and that coverage includes places where traditional data sources (official statistics, census data, household surveys) are sparse, outdated, or politically fraught. A short tour of the literature:

- **Henderson, Storeygard, and Weil (2012)** showed that the intensity of nighttime lights, observed from satellites, is a remarkably good proxy for economic activity. They used this to construct GDP estimates for sub-national regions and countries where official statistics were unreliable. The paper is now a standard reference and has spawned an entire literature on "nightlights economics."

- **Jean et al. (2016)**, published in *Science*, took the next step: they used **daytime** satellite imagery (which contains vastly more information than nightlights) combined with deep learning to predict village-level poverty in five African countries. Critically, they did this with **transfer learning** — starting from an ImageNet-pretrained CNN and fine-tuning it to predict nightlight intensity as a proxy task, then using the resulting features to predict survey-based measures of consumption and wealth. This paper is the direct intellectual motivation for everything we do in this notebook.

- **Donaldson and Storeygard (2016)**, in a *Journal of Economic Perspectives* survey titled "The View from Above," laid out a broader research agenda for remote sensing in economics. They catalog applications in agriculture, urban development, deforestation, conflict, and more.

- **Rolf et al. (2021)** introduced **MOSAIKS**, an approach that precomputes a large bank of random-convolution features from satellite imagery and then fits simple ridge regressions to predict many different outcomes (forest cover, income, housing prices, population density). The trick is that the feature extraction step happens once, and every downstream task is then just a linear regression — extremely fast and easy to apply by non-ML researchers.

A common thread across all of this work is that **labeled data is scarce** — you might have GDP figures for a few hundred districts, or household survey data for a few thousand clusters — but **unlabeled satellite imagery is abundant**. This is exactly the scenario in which foundation models and transfer learning shine.

The rest of this notebook walks through a simplified version of the pipeline: load a pretrained vision model, fine-tune it to classify land use in satellite images, and then discuss how one would go from a land-use classifier to an economic measurement. We use the EuroSAT dataset, which is small enough to run in a classroom setting while still being representative of what real satellite data looks like.

## The EuroSAT Dataset

EuroSAT (Helber et al., 2019) is a collection of ~27,000 Sentinel-2 satellite image patches covering 34 European countries. Each image is 64×64 pixels and is labeled with one of ten land-use classes:

| Class | Description |
|---|---|
| `AnnualCrop` | Agricultural land with annual crops (wheat, corn, etc.) |
| `Forest` | Forested areas |
| `HerbaceousVegetation` | Grasslands, meadows |
| `Highway` | Paved highways and major roads |
| `Industrial` | Industrial and commercial buildings |
| `Pasture` | Grazing land |
| `PermanentCrop` | Vineyards, orchards, olive groves |
| `Residential` | Residential urban areas |
| `River` | Rivers, linear water features |
| `SeaLake` | Seas, lakes, reservoirs |

These classes are not literally "economic activity," but they are strong proxies for different kinds of economic activity: industrial and residential areas reflect population and manufacturing density, crop classes reflect agricultural output, highways reflect transportation infrastructure, and so on. A good land-use classifier is a useful building block for more ambitious economic applications.

Sentinel-2 itself is part of the European Space Agency's Copernicus program. It has 13 spectral bands, but EuroSAT distributes only the three RGB visible bands, which is convenient for us because it means we can reuse ResNet's RGB preprocessing pipeline directly.

In [ ]:
# Load the dataset (downloads on first call).
# EuroSAT is an ImageFolder dataset — each class lives in its own subdirectory.
eurosat_raw = datasets.EuroSAT(root="./data", download=True)
print(f"Total images: {len(eurosat_raw):,}")
print(f"Classes ({len(eurosat_raw.classes)}): {eurosat_raw.classes}")

# Class counts
targets = np.array(eurosat_raw.targets)
print("\nImages per class:")
for i, name in enumerate(eurosat_raw.classes):
    print(f"  {name:<25} {int((targets == i).sum()):>6,}")

Let's look at a few example images from each class. EuroSAT images are small (64×64), and with the naked eye some classes are easy to distinguish while others are visually similar.

In [ ]:
# Grid of 4 examples per class
fig, axes = plt.subplots(10, 4, figsize=(7, 14))
rng = np.random.default_rng(0)
for cls_idx, cls_name in enumerate(eurosat_raw.classes):
    cls_indices = np.where(targets == cls_idx)[0]
    picks = rng.choice(cls_indices, size=4, replace=False)
    for j, p in enumerate(picks):
        img, _ = eurosat_raw[int(p)]
        axes[cls_idx, j].imshow(img)
        axes[cls_idx, j].set_xticks([])
        axes[cls_idx, j].set_yticks([])
        if j == 0:
            axes[cls_idx, j].set_ylabel(cls_name, fontsize=9, rotation=0,
                                         ha="right", va="center")
plt.tight_layout()
plt.show()

A few visual observations:

- **Forest** is uniformly green and textured — easy.
- **SeaLake** and **River** are both blue, but rivers have a thin, elongated shape while lakes/seas are large and uniform.
- **AnnualCrop**, **PermanentCrop**, **Pasture**, and **HerbaceousVegetation** all look like vegetation and are easy to confuse with each other.
- **Residential** has the regular grid pattern of streets and houses; **Industrial** has larger structures and more concrete.
- **Highway** often shows linear paved features.

We should expect the model to do well on the visually distinctive classes and to struggle with the crop classes.

## Data Pipeline

We use the same preprocessing as notebook 2 (ResNet's ImageNet transforms), do a reproducible 80/20 train/test split, and subsample for speed.

In [ ]:
# Reload EuroSAT with ResNet preprocessing
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
eurosat = datasets.EuroSAT(root="./data", transform=preprocess, download=False)

# Reproducible 80/20 split at the *sample* level.
rng = np.random.default_rng(0)
all_indices = np.arange(len(eurosat))
rng.shuffle(all_indices)
split = int(0.8 * len(all_indices))
train_all = all_indices[:split]
test_all = all_indices[split:]

# Subsample for speed: ~300 train / 100 test per class would give 3000/1000.
def subsample_by_class(indices, targets, n_per_class):
    selected = []
    counts = {}
    for idx in indices:
        cls = targets[idx]
        if counts.get(cls, 0) < n_per_class:
            selected.append(int(idx))
            counts[cls] = counts.get(cls, 0) + 1
    return selected


train_idx = subsample_by_class(train_all, targets, n_per_class=300)
test_idx = subsample_by_class(test_all, targets, n_per_class=100)

print(f"Train subset: {len(train_idx):,}")
print(f"Test subset:  {len(test_idx):,}")

train_ds = Subset(eurosat, train_idx)
test_ds = Subset(eurosat, test_idx)

## Baseline: Linear Probe with Pretrained ResNet-18

Exactly the same pattern as notebook 2 — load the pretrained model, freeze the backbone, replace the final `fc` layer with a 10-way head, and train only the head.

In [ ]:
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_one(model, train_ds, test_ds, epochs, lr, batch_size=64):
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(trainable, lr=lr)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model.to(DEVICE)
    losses = []
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * xb.size(0)
            n += xb.size(0)
        losses.append(total / n)
        print(f"  epoch {epoch + 1:2d}  train loss {losses[-1]:.3f}")
    duration = time.time() - t0

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            preds = model(xb).argmax(dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())
    preds = torch.cat(all_preds).numpy()
    labs = torch.cat(all_labels).numpy()
    acc = (preds == labs).mean()
    return losses, acc, duration, preds, labs

In [ ]:
torch.manual_seed(0)
probe = resnet18(weights=weights)
for p in probe.parameters():
    p.requires_grad = False
probe.fc = nn.Linear(512, 10)
for p in probe.fc.parameters():
    p.requires_grad = True

print(f"Linear probe trainable params: {count_trainable(probe):,}")

print("\nTraining linear probe...")
probe_losses, probe_acc, probe_time, probe_preds, probe_labs = train_one(
    probe, train_ds, test_ds, epochs=3, lr=3e-3
)
print(f"\nTest accuracy: {probe_acc:.3f}  ({probe_time:.1f}s)")

### Per-Class Accuracy and the Confusion Matrix

Aggregate accuracy can hide large per-class variation — for EuroSAT we expect Forest and SeaLake to be easy while the crop classes are hard. A confusion matrix makes this concrete.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(probe_labs, probe_preds, labels=range(10))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(eurosat.classes, rotation=45, ha="right")
ax.set_yticklabels(eurosat.classes)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"EuroSAT confusion matrix (linear probe, test acc={probe_acc:.3f})")

for i in range(10):
    for j in range(10):
        val = cm_norm[i, j]
        if val > 0.01:
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    color="white" if val > 0.5 else "black", fontsize=8)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print("\nPer-class accuracy:")
for i, name in enumerate(eurosat.classes):
    print(f"  {name:<25} {cm_norm[i, i]:.3f}")

The diagonal is dominant, which means the classifier is mostly correct. The off-diagonal mass concentrates exactly where we predicted it would: the crop classes (AnnualCrop, HerbaceousVegetation, PermanentCrop, Pasture) mix with each other, and River can sometimes be confused with SeaLake. The pretrained ImageNet features are strong enough for the visually distinctive classes but do not perfectly disentangle the subtle differences between crop types. This kind of per-class diagnosis is an important habit for any applied machine learning work — a number like "83% accuracy" tells you much less than a confusion matrix.

## Full Fine-Tuning

Let's now unfreeze the full model and fine-tune it with a small learning rate. For EuroSAT the difference is often modest — the linear probe is already strong — but the exercise is worth doing to see how full fine-tuning interacts with satellite-specific structure in the data.

In [ ]:
torch.manual_seed(0)
full = resnet18(weights=weights)
full.fc = nn.Linear(512, 10)
for p in full.parameters():
    p.requires_grad = True

print(f"Full fine-tune trainable params: {count_trainable(full):,}")

print("\nFull fine-tuning...")
full_losses, full_acc, full_time, full_preds, full_labs = train_one(
    full, train_ds, test_ds, epochs=3, lr=1e-4
)
print(f"\nTest accuracy: {full_acc:.3f}  ({full_time:.1f}s)")

In [ ]:
# Side-by-side comparison
rows = [
    ("Linear probe",   probe_acc, count_trainable(probe), probe_time),
    ("Full fine-tune", full_acc,  count_trainable(full),  full_time),
]

fig, ax = plt.subplots(figsize=(7, 4))
names = [r[0] for r in rows]
accs = [r[1] for r in rows]
bars = ax.bar(names, accs, color=["#2ca02c", "#ff7f0e"], edgecolor="black", linewidth=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("Test accuracy")
ax.set_title("EuroSAT: linear probe vs. full fine-tune")
for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(),
            f"{a:.3f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

print(f"\n{'Strategy':<18} {'Test acc':>10}  {'Trainable':>12}  {'Wall time':>10}")
for name, acc, params, dur in rows:
    print(f"{name:<18} {acc:>10.3f}  {params:>12,}  {dur:>9.1f}s")

## Interpretability: Where Is the Model Looking? (Grad-CAM)

A 90%+ test accuracy is comforting, but before using a model in any kind of downstream analysis we want to understand *what the model is keying on*. Did it actually learn to detect the visual features we think it did? Or did it latch onto some irrelevant artifact?

**Grad-CAM** (Gradient-weighted Class Activation Mapping; Selvaraju et al., 2017) is a simple and widely used technique for answering this question. The core idea is:

1. Pick the final convolutional layer of the network. Its output feature maps $A^k$ (indexed by channel $k$) still retain spatial structure — each position in the feature map corresponds to a receptive field in the input image.

2. For the class $c$ we care about, compute the gradient of the class score $y^c$ with respect to each feature map:

   $$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}}$$

   This is the mean gradient, which tells us how important each feature map is for predicting class $c$.

3. Form the Grad-CAM map as a weighted sum of the feature maps, followed by a ReLU to keep only the positively contributing regions:

   $$L^c_{\text{Grad-CAM}} = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$$

4. Upsample this low-resolution map back to the input image size and overlay it.

Grad-CAM does not modify the model — it only queries it. We implement it here from scratch using PyTorch's hook mechanism, because it is a nice example of how hooks let you extract intermediate quantities from any model without touching its source code.

In [ ]:
class GradCAM:
    """Grad-CAM implementation using forward and backward hooks.

    Call ``.attach(module)`` on the convolutional layer you want to probe
    (typically the last conv layer in the network), then call ``.explain(x, class_idx)``
    to get a Grad-CAM heatmap.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        # Forward hook: save the activations.
        self.fwd_handle = target_layer.register_forward_hook(self._save_activations)
        # Backward hook: save the gradients flowing back through this layer.
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        # grad_out is a tuple; we want the gradient w.r.t. the layer's output.
        self.gradients = grad_out[0].detach()

    def explain(self, x, class_idx):
        self.model.zero_grad()
        logits = self.model(x)
        # Scalar score we differentiate: the logit for the target class.
        score = logits[0, class_idx]
        score.backward()

        # alpha_k: global-average-pooled gradient per channel.
        alphas = self.gradients.mean(dim=(2, 3), keepdim=True)
        # Weighted sum of activation maps, then ReLU.
        cam = F.relu((alphas * self.activations).sum(dim=1, keepdim=True))
        # Normalize to [0, 1] for display.
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam.squeeze().cpu().numpy()

    def remove(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()


# Attach Grad-CAM to the last conv layer of the fine-tuned ResNet.
full.eval()
cam = GradCAM(full, full.layer4[-1].conv2)

In [ ]:
# Pick one test image per class and visualize the Grad-CAM overlay.
def unnormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).reshape(3, 1, 1)
    return (img.cpu() * std + mean).clamp(0, 1)


# Find one image per class that the model predicts correctly.
pick_indices = {}
for idx in range(len(test_ds)):
    xi, yi = test_ds[idx]
    if int(yi) not in pick_indices:
        with torch.no_grad():
            pred = full(xi.unsqueeze(0).to(DEVICE)).argmax(dim=1).item()
        if pred == int(yi):
            pick_indices[int(yi)] = idx
    if len(pick_indices) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, cls in zip(axes.flat, range(10)):
    idx = pick_indices[cls]
    img, label = test_ds[idx]
    img_batch = img.unsqueeze(0).to(DEVICE)
    heatmap = cam.explain(img_batch, cls)

    # Upsample heatmap to image size (224).
    heatmap_t = torch.from_numpy(heatmap).unsqueeze(0).unsqueeze(0).float()
    heatmap_up = F.interpolate(heatmap_t, size=(224, 224), mode="bilinear",
                                align_corners=False)[0, 0].numpy()

    base_img = unnormalize(img).permute(1, 2, 0).numpy()
    ax.imshow(base_img)
    ax.imshow(heatmap_up, cmap="jet", alpha=0.45)
    ax.set_title(eurosat.classes[cls], fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Grad-CAM heatmaps overlaid on EuroSAT samples", fontsize=13)
plt.tight_layout()
plt.show()

cam.remove()

For each class, the hot (red) regions of the heatmap show where the model is placing most of its attention when predicting that class. On well-performing classes like Forest, Residential, and Industrial, the heatmap typically covers the bulk of the image uniformly — the model is using the whole texture, as we would hope. On classes with distinctive localized features (Highway, River) the heatmap often concentrates along the linear feature itself. And on visually ambiguous classes the heatmaps are diffuse.

For an economist using these model outputs in a downstream analysis, Grad-CAM is a tool for checking that the model's "reasons" are the ones you'd want to invoke in a paper. If your nightlight-prediction model turns out to be keying on roads, that matters for how you interpret the resulting estimates. You can't include a Grad-CAM panel in every regression, but running it on a handful of examples is a cheap sanity check.

## From Land-Use Classification to Economic Measurement

The model we just trained is a land-use classifier, not a GDP estimator. But the gap between the two is mostly a matter of plumbing. Here is the skeleton of what Jean et al. (2016) and subsequent papers do:

1. **Collect unlabeled imagery.** Sentinel-2 provides global, regular, free satellite imagery. For a target country or region, download tiles covering the administrative units you care about (e.g., districts, counties, grid cells).

2. **Extract features.** Run each tile through a pretrained (or fine-tuned) vision model and save the penultimate-layer embedding. For ResNet-18 this is a 512-dimensional vector per tile. For a large area with thousands of tiles, this step is the main compute cost and is embarrassingly parallel.

3. **Aggregate to administrative units.** Average (or otherwise combine) the tile-level features for all tiles within each administrative unit, producing one feature vector per unit.

4. **Train a supervised predictor.** Using the features as regressors and any available labeled data (household surveys, nightlights, census statistics) as outcomes, fit a simple linear model or ridge regression:

   $$\hat{y}_i = \beta^\top \phi_i + \epsilon_i$$

   where $\phi_i$ is the aggregated feature vector for unit $i$ and $y_i$ is the outcome of interest (e.g., log household consumption). Because $\phi$ is relatively low-dimensional compared to raw imagery, this regression can be fit on a few hundred or a few thousand labeled observations.

5. **Predict out-of-sample.** Use the fitted $\beta$ to generate predictions for units where no labeled data exists. These predictions are the researcher's "satellite-based GDP estimate" or "satellite-based poverty index."

The quality of the final predictions depends on every step: how good the pretrained features are, how representative the labeled training units are of the target units, and how much of the variation in the outcome is actually visible from space. Papers in this area spend considerable effort on **validation** — comparing satellite-based estimates to independent measurements like census data, phone metadata, or consumption surveys — because the whole point of the exercise is to be trusted in places where no ground truth is available.

### Caveats

Satellite-based economic measurement is powerful but is not a free lunch. A few caveats worth stating explicitly:

- **Domain shift.** A model pretrained on European satellite imagery (EuroSAT is collected over Europe) may produce different representations when applied to, say, Sub-Saharan Africa — the visual characteristics of "rural" and "urban" look different in different parts of the world. Transferring across regions requires either retraining or explicit domain adaptation.

- **Reproducibility.** Satellite image datasets are expensive and often proprietary. The pretrained models themselves can change over time. Papers need to document which pretraining, which preprocessing, and which fine-tuning runs were used, and ideally share their trained models.

- **Ethics and power.** Satellite-based estimates enable fine-grained measurement of populations who did not consent to being measured. This is particularly sensitive when the measurements are used for targeting (humanitarian aid, infrastructure investment, surveillance). Several applied ML ethics papers have argued that researchers should be explicit about who benefits from the resulting estimates and who bears the risks.

- **Ground truth is still needed.** All of these methods are trained on labeled data — survey data, census data, nightlights. The satellite imagery doesn't replace the need for ground truth; it multiplies the value of whatever ground truth you already have.

None of these concerns makes the approach less useful, but they do make it a topic that rewards careful applied work.

## Summary

- Satellite imagery is an increasingly important data source in empirical economics, especially in data-poor regions where official statistics are unreliable. Jean et al. (2016) is a canonical example.
- The **EuroSAT dataset** is a small but realistic land-use classification problem built from Sentinel-2 RGB imagery. It is well-suited for classroom exercises because it downloads quickly and trains fast.
- A **linear probe** on ImageNet-pretrained ResNet-18 already achieves strong accuracy on EuroSAT. Full fine-tuning adds a bit more but at a much higher parameter cost.
- The confusion matrix shows that per-class accuracy varies a lot — Forest and SeaLake are easy, crop types are harder — and this kind of diagnostic is essential before using model outputs in downstream analysis.
- **Grad-CAM** is a simple technique for visualizing which image regions drive the model's predictions, implemented in ~30 lines of PyTorch using hooks. It provides the kind of sanity check that should accompany any applied use of deep learning.
- The pipeline from *land-use classification* to *economic measurement* involves feature extraction, aggregation to administrative units, and a classical regression on top — the same pipeline Jean et al. (2016) and subsequent papers use at scale.

## References

- Helber, P., Bischke, B., Dengel, A., & Borth, D. (2019). EuroSAT: A novel dataset and deep learning benchmark for land use and land cover classification. *IEEE JSTARS*.
- Henderson, J. V., Storeygard, A., & Weil, D. N. (2012). Measuring economic growth from outer space. *American Economic Review*, 102(2), 994–1028.
- Jean, N., Burke, M., Xie, M., Davis, W. M., Lobell, D. B., & Ermon, S. (2016). Combining satellite imagery and machine learning to predict poverty. *Science*, 353(6301), 790–794.
- Donaldson, D., & Storeygard, A. (2016). The view from above: Applications of satellite data in economics. *Journal of Economic Perspectives*, 30(4), 171–198.
- Rolf, E., Proctor, J., Carleton, T., Bolliger, I., Shankar, V., Ishihara, M., Recht, B., & Hsiang, S. (2021). A generalizable and accessible approach to machine learning with global satellite imagery (MOSAIKS). *Nature Communications*, 12(1), 1–11.
- Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. *ICCV 2017*.
